In [1]:
import json
from monty.json import MontyDecoder
import os
from pymatgen.core import Element

from utils_kga.statistical_analysis.analyze_ion_pairs import *

In [2]:
# Load edge-df
with open("data/dfs_of_magnetic_edge_information.json") as f:
# replace to analyze all structs, not only cryst. uniques:
#with open("data/dfs_of_magnetic_edge_information_include_crystallographic_multiples.json") as f:
    dict_all_stats = json.load(f)
all_stats = {key: pd.DataFrame.from_dict(df) for key, df in dict_all_stats.items()}

# For metadata filtering and computation of occurrences in magnetic primitive cells
with open("../../data_retrieval_and_preprocessing_MAGNDATA/data/df_grouped_and_chosen_commensurate_MAGNDATA.json") as f:
    df = json.load(f, cls=MontyDecoder)

plot_dir = "plots/TM_octahedra_ion_pair_distributions_at_large_bond_angles"
os.makedirs(plot_dir, exist_ok=True)

In [3]:
for ang_df in all_stats.values():
    ang_df["site_is_tm"] = ang_df["site_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["site_to_is_tm"] = ang_df["site_to_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["ligand_el_set"] = ang_df["ligand_elements"].apply(lambda ls: set(ls))

    ang_df["bond_angle_strict_180"] = ang_df["mag_ligand_mag_angle"].apply(lambda ls: len(ls) == 1 and round(ls[0], 4) == 180.0)
    ang_df["bond_angle_greater_150"] = ang_df["mag_ligand_mag_angle"].apply(lambda ls: len(ls) == 1 and round(ls[0], 4) > 150.0)


In [4]:
write_mode = "w"
ba_threshold = 150
normalize_bool = False
normalize_string = "absolute occurrences"

data_string = f"greater {ba_threshold}° bond angles with TM octahedra at both nodes"
print(data_string)
occus = {"afm": {}, "fm": {}}
num_structures = {}  # track in how many structures ion pair is found with these conditions

for md_id, ang_df in all_stats.items():
    subdf = ang_df.loc[(ang_df["site_is_tm"])  & (ang_df["site_to_is_tm"])]
    subdf = subdf.loc[subdf[f"bond_angle_greater_{ba_threshold}"]]
    subdf = subdf.loc[(subdf["site_ce"]=="O:6")
            & (subdf["site_to_ce"]=="O:6")]

    if not subdf.empty:
        n_lattice_points = df.at[md_id, "n_lattice_points"]
        for mag_type, condition in zip(["afm", "fm", ],
                                        [(subdf["spin_angle"]>170), (subdf["spin_angle"]<10)]):
            type_df = subdf.loc[condition]
            if not type_df.empty:
                abs_ion_pairs = get_ion_pair_occus(df=type_df, n_lattice_points=n_lattice_points, normalize_bool=normalize_bool)

                for k, v in abs_ion_pairs.items():
                    # Assert no detected ions are missing
                    assert (k[0], k[1]) in sort_by_electron_configuration, str((k[0], k[1]))
                    assert (k[2], k[3]) in sort_by_electron_configuration, str((k[2], k[3]))
                    if k in occus[mag_type]:
                        occus[mag_type][k] += v
                    else:
                        occus[mag_type][k] = v

                    if k in num_structures:
                        num_structures[k].add(md_id)
                    else:
                        num_structures[k] = {md_id}



print(occus)

figure_dict = plot_ion_pair_occurrences(occus=occus, log=False)
for mag_type, fig in figure_dict.items():
    print(mag_type)
    fig.update_layout(title=dict(
        text=f"{mag_type} occurrences, {data_string}, {normalize_string}",
        font=dict(size=10)))
    # fig.show()
    fig.write_html(os.path.join(plot_dir, f"ion_pair_{normalize_string.replace(' ', '_')}_{mag_type}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))
    fig.write_image(os.path.join(plot_dir, f"ion_pair_{normalize_string.replace(' ', '_')}_{mag_type}_{data_string.replace(' ', '_').replace('°', 'deg')}.pdf"))
    pass

# Plot heatmap of FM/AFM ratio per ion pair (logarithmic, diverging color scale)
ratio_fig = plot_ion_pair_occurrence_ratio(occus=occus)
ratio_fig.update_layout(title=dict(
    text=f"Logarithmic plot of FM/AFM ratios, {data_string}, {normalize_string}",
    font=dict(size=10)))
# ratio_fig.show()
ratio_fig.write_html(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))
ratio_fig.write_image(os.path.join(plot_dir, f"log_plot_afm-fm-ratio_ion_pair_{normalize_string.replace(' ', '_')}_{data_string.replace(' ', '_').replace('°', 'deg')}.pdf"))

close_to_1_tracker, all_tracker = set(), set()
print("FM to AFM ratio  :")

for ip in occus["fm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if ip in occus["afm"]:
        print(ip, round(occus["fm"][ip] / occus["afm"][ip], 2), round(log10(occus["fm"][ip] / occus["afm"][ip]), 2), len(num_structures[ip]))
        if abs(log10(occus["fm"][ip] / occus["afm"][ip])) < 0.5:
            close_to_1_tracker.add(ip_string)
        all_tracker.add(ip_string)
    else:
        print(ip, "-- (only FM)", len(num_structures[ip]))
        all_tracker.add(ip_string)
for ip in occus["afm"]:
    ip_string = "".join(sorted([str(ip[0])+str(ip[1]), str(ip[2])+str(ip[3])]))
    if not ip in occus["fm"]:
        print(ip, "-- (only AFM)", len(num_structures[ip]))
        all_tracker.add(ip_string)

print(len(close_to_1_tracker), len(all_tracker), len(close_to_1_tracker)/len(all_tracker))

print("--------------- num structures ------------------------")
print(num_structures)
n = set()
for s in num_structures.values():
    n.update(s)
print("Num structures: ", len(n))

# Display num structures in similar plot
numstructfig = plot_ion_pair_occurrences(occus={"num_structs": {k: len(v) for k, v in num_structures.items()}}, log=False)["num_structs"]
numstructfig.update_layout(title=dict(
    text=f"Number of structures, {data_string}, {normalize_string}",
    font=dict(size=10)))
# numstructfig.show()
numstructfig.write_html(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.html"))
numstructfig.write_image(os.path.join(plot_dir, f"num_structures_ion_pair_{data_string.replace(' ', '_').replace('°', 'deg')}.pdf"))

greater 150° bond angles with TM octahedra at both nodes
{'afm': {('Mn', 3, 'Mn', 3): 32.0, ('Cr', 3, 'Cr', 3): 184.0, ('Cu', 2, 'Cu', 2): 132.0, ('Co', 2, 'Os', 6): 32.0, ('Os', 6, 'Co', 2): 32.0, ('Mn', 4, 'Mn', 4): 36.0, ('Os', 5, 'Os', 5): 24.0, ('Fe', 3, 'Fe', 3): 134.0, ('Co', 3, 'Co', 3): 48.0, ('Mn', 2, 'Mn', 2): 64.0, ('Ni', 2, 'Ni', 2): 136.0, ('Fe', 3, 'Os', 5): 40.0, ('Os', 5, 'Fe', 3): 40.0, ('Ru', 4, 'Ru', 4): 16.0, ('Co', 2, 'Ru', 5): 16.0, ('Ru', 5, 'Co', 2): 16.0, ('V', 3, 'V', 3): 16.0, ('Ni', 2, 'Os', 6): 16.0, ('Os', 6, 'Ni', 2): 16.0, ('Fe', 2, 'Fe', 3): 104.0, ('Fe', 3, 'Fe', 2): 104.0, ('Ir', 4, 'Ir', 4): 32.0, ('Co', 2, 'Co', 2): 64.0, ('Ni', 3, 'Ni', 3): 72.0, ('Fe', 2, 'Fe', 2): 48.0, ('Ni', 2, 'Ir', 4): 16.0, ('Ir', 4, 'Ni', 2): 16.0}, 'fm': {('Mn', 3, 'Mn', 3): 104.0, ('Cu', 2, 'Cu', 2): 36.0, ('Co', 2, 'Os', 6): 40.0, ('Os', 6, 'Co', 2): 40.0, ('Fe', 2, 'Fe', 3): 40.0, ('Fe', 3, 'Fe', 2): 40.0, ('Cr', 3, 'Cr', 3): 20.0, ('Ru', 4, 'Ru', 4): 72.0, ('Mn', 4, '